# Análise Exploratória De Dados (EDA)

Objetivo desta fase é entender as features do dataset de clientes da Calcomm

## 1 - Configuração Do Ambiente

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
# Bibliotecas básicas
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Estatística
from scipy import stats

## 2 - Características do dataset bruto

O dataset usado para a experimentação foi elaborado através de fontes de dados internas de operações e vendas da Calcomm.

### Features disponíveis

O dataset bruto apresenta 33 colunas de dados de identificação dos clientes, demografia e características de serviço.

- CustomerID: código de identificação do cliente
- Count: Quantidade por cliente, em geral 1
- Country: país
- State: estado
- City: cidade
- Zip Code: CEP do endereço
- Lat Long: latitude e longitude combinadas do endereço do cliente
- Latitude: latitude do endereço do cliente
- Longitude: longitude do endereço do cliente
- Gender: sexo do cliente (m/f)
- Senior Citizen: a partir de 65 anos (s/n)
- Partner: Possui companheiro (s/n)
- Dependents: Possui dependentes (s/n)
- Tenure Months: tempo de serviço (meses)
- Phone Service: Possui serviço de telefonia (s/n)
- Multiple Lines: possui múltiplas linhas telefônicas (s/n)
- Internet Service: Possui serviço de internet (s/n)
- Online Security: Possui serviço de segurança online (s/n)
- Online Backup: Possui serviço de backup online (s/n)
- Device Protection: Possui serviço de proteção de dispositivo (s/n)
- Tech Support: Possui serviço suporte técnico (s/n)
- Streaming TV: Possui serviço de streaming de programas de TV (s/n)
- Streaming Movies: Possui serviço de streaming de filmes (s/n)
- Contract: tipo de contrato de fidelidade (mensal, anual ou bienal)
- Paperless Billing: fatura digital (s/n)
- Payment Method: método de pagamento (débito automático, cartão de crédito ou pix)
- Monthly Charges: custo menal (número, floating)
- Total Charges: despesas trimestrais (número, floating)
- Churn Label: indicador de churn, cliente deixou a Calcomm no trimestre (s/n)
- Churn Value: indicador numérico de churn (0/1)
- Churn Score: indicador numérico de probabilidade de saída (churn) (0-100)
- CLTV: Customer Lifetime Value (numérico)
- Churn Reason: motivo da saída (string)



### Observações

Obs 1
Obs 2


In [0]:
# Aponta novo dataset com dados brutos
path = "../data/raw/Telco_customer_churn.csv"

# Carrega dataset em um dataframe pandas
df = pd.read_csv(path)

# Exibe informações sobre o dataset
print(f"Shape dos dados: {df.shape}")
print(f"Nome das colunas:\n{df.columns.tolist()}")

# Exibe primeiras linhas do dataset
df.head(10)

Os dados brutos  mostram ...

In [0]:
# Informações gerais sobre o dataset
print("=== INFORMAÇÕES GERAIS DO DATASET ===\n")
print(df.info())

print("\n=== ESTATÍSTICAS DESCRITIVAS ===\n")
df.describe()

## 3 - Análise exploratória

### 3.1 - Valores ausentes no dataset

O EDA exclui cálculo da coluna de motivo do churn quando não ocorreu o churn.

In [0]:
# Análise de valores ausentes
print("=== ANÁLISE DE MISSING VALUES ===\n")

missing_values = pd.DataFrame({
    'Coluna': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})

missing_values = missing_values[missing_values['Missing_Count'] > 0].sort_values(
    by='Missing_Percentage', ascending=False
)

# 'Churn Reason' só é esperado para clientes que fizeram churn.
# Recalcula o missing apenas sobre a base de churners para não inflar o percentual.
if 'Churn Reason' in missing_values.index:
    churners = df[df['Churn Label'] == 'Yes']
    reason_missing = churners['Churn Reason'].isnull().sum()
    reason_pct = round(reason_missing / len(churners) * 100, 2)
    missing_values.loc['Churn Reason', 'Missing_Count'] = reason_missing
    missing_values.loc['Churn Reason', 'Missing_Percentage'] = reason_pct
    missing_values = missing_values[missing_values['Missing_Count'] > 0].sort_values(
        by='Missing_Percentage', ascending=False
    )

if len(missing_values) > 0:
    print(missing_values)
    print("\nNota: o percentual de 'Churn Reason' é calculado apenas sobre churners (Churn Label = Yes).")
    
    # Visualizar missing values
    plt.figure(figsize=(10, 6))
    plt.barh(missing_values['Coluna'], missing_values['Missing_Percentage'], color='coral')
    plt.xlabel('Porcentagem de Missing Values (%)')
    plt.title('Distribuição de Missing Values por Coluna')
    plt.tight_layout()
    plt.show()
else:
    print("Nenhum missing value detectado!")

In [0]:
# Substitua valores em branco ' ' na colune "Total Charge" por 0.0 

### 3.2 - Análise da variável target

In [0]:
# Análise da variável alvo - Churn

# Renomear variável `Churn Value` para `target`
df.rename(columns={'Churn Value': 'target'}, inplace=True)

# Converter target para binário (0 = não houve chrun, 1 = houve churn)
# No dataset original, valores > 0 indicam presença de doença
df['target'] = (df['target'] > 0).astype(int)

print("=== DISTRIBUIÇÃO DA VARIÁVEL TARGET ===\n")
target_counts = df['target'].value_counts()
target_percentages = df['target'].value_counts(normalize=True) * 100

print("Contagem:")
print(target_counts)
print("\nPercentual:")
for idx, pct in target_percentages.items():
    label = "Churn não ocorreu" if idx == 0 else "Houve churn"
    print(f"{label} ({idx}): {pct:.2f}%")

# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
axes[0].bar(['Churn não ocorreu', 'Houve churn'], target_counts.values, color=['lightgreen', 'coral'])
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição da Variável Target')
axes[0].grid(axis='y', alpha=0.3)

# Gráfico de pizza
axes[1].pie(target_counts.values, labels=['Churn não ocorreu', 'Houve churn'], 
            autopct='%1.1f%%', colors=['lightgreen', 'coral'], startangle=90)
axes[1].set_title('Proporção de Classes')

plt.tight_layout()
plt.show()

# Verificar se há desbalanceamento
ratio = target_counts.min() / target_counts.max()
print(f"\nRatio de balanceamento: {ratio:.2f}")
if ratio < 0.5:
    print("⚠️ Dataset desbalanceado! Considere usar técnicas como SMOTE ou class_weight.")
else:
    print("✓ Dataset razoavelmente balanceado.")

### 3.3 - Análise De Outliers

A análise de outliers permite visualizar dados com valores extremos que podem ser válidos, erros de entrada ou coleta e ajudar na decisão sobre tratamento adicional de um ou mais deles, como normalização ou exclusão.

**Métodos utilizados:**

- **Z-Score**: Identifica valores que estão a mais de 3 desvios padrão da média
- **Visualização**: Mostra boxplots para identificação visual


In [0]:
# Colunas numéricas (excluindo CustomerID e target que é a variável alvo). A feature "count" também não deve ser usada na análise.
numeric_cols = df.select_dtypes(include=[np.number]).columns
numeric_cols = numeric_cols.drop(['CustomerID', 'target', "Count"], errors='ignore')

n = len(numeric_cols)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4 * nrows))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    sns.boxplot(x=df[col], ax=axes[idx], color='coral')
    axes[idx].set_title(f'Boxplot: {col}', fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].grid(axis='x', alpha=0.3)
    col_zscore = np.abs(stats.zscore(df[col].dropna()))
    outlier_count = (col_zscore > 3).sum()
    axes[idx].text(0.95, 0.95, f'Outliers: {outlier_count}', 
                   transform=axes[idx].transAxes, fontsize=9,
                   verticalalignment='top', horizontalalignment='right',
                   bbox=dict(facecolor='white', alpha=0.5, edgecolor='gray'))

for ax in axes[n:]:
    fig.delaxes(ax)

plt.tight_layout()
plt.show()


**Análise inicial**: Como os dados são do estado da Califórnia, os valores relacionados à localidade não deveriam apresentar outliers como indicado nos boxplot acima. O dataset trás valores de coordenadas geográficas e CEP (ZIP) similares. Caso houvesse outliers, seria necessário identificar se são estatísticos pelo modelo de distribuião de ZIP codes ou extensão territorial do estado ou se poderiam ser dados errados como um ZIP code ou coordenada distante, possivelmente erro de coleta ou entrada de dados. A análise de anomalias a seguir também contribui para esta verificação.




### 3.4 - Análise De Anomalias E Valores Inválidos

**Objetivo**: Encontrar valores em features fora de contexto na relação de churn.


In [0]:
print("=== ANÁLISE DE ANOMALIAS E VALORES INVÁLIDOS ===\n")

# Verificações de consistência baseadas em conhecimento de domínio
anomalies = []

# 1. ZIP code (deve estar entre 90000 e 96162 de acordo com https://www.unitedstateszipcodes.org/ca/)
zip_code = df[(df['Zip Code'] < 90000) | (df['Zip Code'] > 96162)]
if len(zip_code) > 0:
    anomalies.append(f"ZIP Code fora do intervalo válido para Califórnia: {len(zip_code)} casos")
    
# 2. Churn Score (ChurnScore) - valores válidos: 0-100
churn_score = df[(df['Churn Score'] < 0) | (df['Churn Score'] > 100)]
if len(churn_score) > 0:
    anomalies.append(f"Churn Score anormal: {len(churn_score)} casos")
    print(f"⚠️ Churn Score anormal detectada: {len(churn_score)} casos")
    print(f"   Valores: min={df['churn_score'].min()}, max={df['churn_score'].max()}")

# # 3. Colesterol (chol) - valores típicos: 100-400 mg/dl
# chol_anomalies = df[(df['chol'] < 100) | (df['chol'] > 400)]
# if len(chol_anomalies) > 0:
#     anomalies.append(f"Colesterol anormal: {len(chol_anomalies)} casos")
#     print(f"⚠️ Colesterol anormal detectado: {len(chol_anomalies)} casos")
#     print(f"   Valores: min={df['chol'].min()}, max={df['chol'].max()}")

# # 4. Frequência cardíaca máxima (thalch) - valores típicos: 60-220 bpm
# hr_anomalies = df[(df['thalch'] < 60) | (df['thalch'] > 220)]
# if len(hr_anomalies) > 0:
#     anomalies.append(f"Frequência cardíaca anormal: {len(hr_anomalies)} casos")
#     print(f"⚠️ Frequência cardíaca anormal detectada: {len(hr_anomalies)} casos")
#     print(f"   Valores: min={df['thalch'].min()}, max={df['thalch'].max()}")

# 5. Variáveis categóricas com valores fora do esperado
categorical_checks = {
    'Gender': ["Female", "Male"],
    'Senior Citizen': ["Yes", "No"],
    'Partner': ["Yes", "No"],
    'Dependents': ["Yes", "No"],
    'Phone Service': ["Yes", "No"],
    'Multiple Lines': ["Yes", "No", "No phone service"],
    'Internet Service': ["DSL", "Fiber optic", "No"],
    'Online Security': ["Yes", "No", "No internet service"],
    'Online Backup': ["Yes", "No", "No internet service"],
    'Device Protection': ["Yes", "No", "No internet service"],
    'Tech Support': ["Yes", "No", "No internet service"],
    'Streaming TV': ["Yes", "No", "No internet service"],
    'Streaming Movies': ["Yes", "No", "No internet service"],
    'Contract': ["Month-to-month", "One year", "Two year"],
    'Paperless Billing': ["Yes", "No"],
    'Payment Method': ["Bank transfer (automatic)", "Credit card (automatic)", "Electronic check", "Mailed check"],
}

for col, valid_values in categorical_checks.items():
    if col in df.columns:
        invalid = df[~df[col].isin(valid_values) & df[col].notna()]
        if len(invalid) > 0:
            anomalies.append(f"{col}: {len(invalid)} valores inválidos")
            print(f"⚠️ {col}: {len(invalid)} valores fora do domínio esperado {valid_values}")

if len(anomalies) == 0:
    print("✓ Nenhuma anomalia óbvia detectada nas validações de domínio!")
else:
    print(f"\n📊 Total de tipos de anomalias detectadas: {len(anomalies)}")

# Verificar duplicados
duplicates = df.duplicated().sum()
print(f"\n=== DUPLICADOS ===")
print(f"Registros duplicados: {duplicates}")
if duplicates > 0:
    print("⚠️ Considere remover ou investigar registros duplicados")

### 3.5 Análise de Distribuições

**Objetivo:** Entender a distribuição das variáveis numéricas e identificar assimetrias

In [0]:
# Análise de distribuições
print("=== ANÁLISE DE ASSIMETRIA (SKEWNESS) E CURTOSE ===\n")

distribution_stats = []
for col in numeric_cols:
    skewness = df[col].skew()
    kurtosis = df[col].kurtosis()
    distribution_stats.append({
        'Coluna': col,
        'Skewness': round(skewness, 3),
        'Kurtosis': round(kurtosis, 3),
        'Interpretação': 'Normal' if abs(skewness) < 0.5 else ('Assimétrica à direita' if skewness > 0 else 'Assimétrica à esquerda')
    })

dist_df = pd.DataFrame(distribution_stats)
print(dist_df.to_string(index=False))

# Visualizar distribuições
fig, axes = plt.subplots(3, 5, figsize=(18, 12))
axes = axes.ravel()

for idx, col in enumerate(numeric_cols):
    if idx < len(axes):
        axes[idx].hist(df[col].dropna(), bins=30, color='skyblue', edgecolor='black', alpha=0.7)
        axes[idx].set_title(f'{col}\nSkew: {df[col].skew():.2f}', fontsize=9, fontweight='bold')
        axes[idx].set_xlabel('Valor')
        axes[idx].set_ylabel('Frequência')
        axes[idx].grid(axis='y', alpha=0.3)

# Remover subplots vazios
for idx in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[idx])

plt.suptitle('Distribuições das Variáveis Numéricas', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

### 3.6 Análise de Correlações

**Objetivo:** Identificar relações entre variáveis e detectar multicolinearidade


In [0]:
# Matriz de correlação apenas para colunas numéricas
corr_numeric = df.select_dtypes(include=[np.number]).corr()
corr_numeric = corr_numeric.drop(["Count"], errors='ignore')

# Visualizar matriz de correlação
plt.figure(figsize=(14, 10))
sns.heatmap(corr_numeric, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlação - Churn em serviços telecom', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Identificar correlações fortes com o target
print("=== CORRELAÇÕES COM A VARIÁVEL TARGET ===\n")
target_corr = corr_numeric['target'].sort_values(ascending=False)
print(target_corr)

# Identificar pares de features com alta correlação (possível multicolinearidade)
print("\n=== MULTICOLINEARIDADE (Correlação entre features) ===")
print("Pares de features com correlação > 0.7:\n")

high_corr_pairs = []
for i in range(len(corr_numeric.columns)):
    for j in range(i+1, len(corr_numeric.columns)):
        if abs(corr_numeric.iloc[i, j]) > 0.7 and corr_numeric.columns[i] != 'target' and corr_numeric.columns[j] != 'target':
            high_corr_pairs.append({
                'Feature 1': corr_numeric.columns[i],
                'Feature 2': corr_numeric.columns[j],
                'Correlação': round(corr_numeric.iloc[i, j], 3)
            })

if len(high_corr_pairs) > 0:
    high_corr_df = pd.DataFrame(high_corr_pairs).sort_values(by='Correlação', ascending=False)
    print(high_corr_df.to_string(index=False))
    print("\n⚠️ Alta correlação entre features pode causar multicolinearidade!")
else:
    print("✓ Nenhuma correlação forte detectada entre features (excluindo target)")

**Observaçõe**:
- Algumas correlações podem não fazer sentido como Zip Code com Latitude e Longitude.

## 4. Preparação dos Dados

### 4.1 Tratamento de Missing Values

**Estratégias utilizadas:**
- Para variáveis numéricas: imputação pela mediana (mais robusta a outliers)
- Para variáveis categóricas: imputação pela moda ou categoria específica
- Análise do impacto da imputação

> **Data Leakage:** As estatísticas de imputação (mediana, moda) devem ser calculadas **somente nos dados de treino** e depois aplicadas ao conjunto de teste. Calcular essas estatísticas no dataset completo "vaza" informação do teste para o treino, gerando uma estimativa otimista do desempenho real do modelo.

In [0]:
from sklearn.model_selection import train_test_split

print("=== TRATAMENTO DE MISSING VALUES ===")

# Copiar o dataframe original para preservar os dados
df_clean = df.copy()

# Divisão prévia em treino e teste para evitar data leakage:
# as estatísticas de imputação são calculadas SOMENTE nos dados de treino
# e depois aplicadas ao conjunto de teste.
# test_size=0.2 reserva 20% dos dataset para testar o modelo
X_raw = df_clean.drop(columns=["target"])
y_raw = df_clean["target"]
X_train_raw, X_test_raw, _, _ = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# Imputação para variáveis numéricas: mediana calculada SOMENTE no treino
# Não há variáveis numéricas com dados faltantes nesse dataset, entretanto ele estará preparado para
# inputar algumas em treinamento adicional.
num_missing = ["trestbps", "chol", "thalch", "oldpeak", "ca"]
for col in num_missing:
    if col in X_train_raw.columns:
        median = X_train_raw[col].median()
        df_clean[col] = df_clean[col].fillna(median)
        print(f"{col}: imputado com mediana do treino ({median:.2f})")

# Imputação para variáveis categóricas: moda calculada SOMENTE no treino
cat_missing = ["Zip Code", "Latitude", "Longitude", "Monthly Charge", "Total Charge"]
for col in cat_missing:
    if col in X_train_raw.columns:
        mode = X_train_raw[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(mode)
        print(f"{col}: imputado com moda do treino ('{mode}')")

print("✓ Tratamento de missing values concluído!")
print(f"Shape após imputação: {df_clean.shape}")
print("Valores ausentes restantes por coluna:")
print(df_clean.isnull().sum())

## 4.2 Codificação de Variáveis Categóricas (One-hot Encoding)

Para que algoritmos de machine learning possam trabalhar com variáveis categóricas, é necessário convertê-las em formato numérico. O método mais comum é o **One-hot Encoding**, que transforma cada categoria em uma coluna binária (0 ou 1).



**Passos:**
- Identifique as colunas categóricas relevantes, listadas abaixo
- Aplique o `pd.get_dummies()` para criar as variáveis dummies.
- Evite a duplicidade de informação removendo uma categoria de cada variável (drop_first=True), se necessário.

    - 'Gender': ["Female", "Male"]
    - 'Senior Citizen': ["Yes", "No"]
    - 'Partner': ["Yes", "No"]
    - 'Dependents': ["Yes", "No"]
    - 'Phone Service': ["Yes", "No"]
    - 'Multiple Lines': ["Yes", "No", "No phone service"]
    - 'Internet Service': ["DSL", "Fiber optic", "No"]
    - 'Online Security': ["Yes", "No", "No internet service"]
    - 'Online Backup': ["Yes", "No", "No internet service"]
    - 'Device Protection': ["Yes", "No", "No internet service"]
    - 'Tech Support': ["Yes", "No", "No internet service"]
    - 'Streaming TV': ["Yes", "No", "No internet service"]
    - 'Streaming Movies': ["Yes", "No", "No internet service"]
    - 'Contract': ["Month-to-month", "One year", "Two year"]
    - 'Paperless Billing': ["Yes", "No"]
    - 'Payment Method': ["Bank transfer (automatic)", "Credit card (automatic)", "Electronic check", "Mailed check"]


**Observações:**
- O One-hot Encoding aumenta o número de colunas, mas permite que modelos interpretem corretamente as categorias.
- Após a codificação, utilize o novo dataframe (`df_encoded`) para as etapas de modelagem.

In [0]:
# Remover a coluna `dataset`, 'Country' e 'State' pois não são relevantes para prever churn
# City causa alta cardinalidade no dataset, cerca de 1100 colunas em one-hot enconding.
df_clean.drop(columns=['dataset', 'Country', 'State', 'City', 'Lat Long'], inplace=True, errors='ignore')

# One-hot encoding das variáveis categóricas relevantes
categorical_cols = ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method', "Churn Label", "Churn Reason"]
df_encoded = pd.get_dummies(df_clean, columns=categorical_cols, drop_first=True)

print(f"Shape após one-hot encoding: {df_encoded.shape}")
df_encoded.head()

## 5. Experimentação e MVP (Minimum Viable Product)

### Experimento: Regressão Logística (Baseline)
Treine um modelo baseline e registre no MLFlow

In [0]:
# Divisão treino/teste com os mesmos parâmetros da imputação (seção 4.1),
# garantindo consistência entre os conjuntos utilizados no pré-processamento e na modelagem.
X = df_encoded.drop(columns=["CustomerID", "target"])
y = df_encoded["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
pip install mlflow

In [0]:
%pip install --upgrade typing_extensions mlflow
import sys
for name in list(sys.modules):
    if name == 'mlflow' or name.startswith('mlflow.') or name == 'typing_extensions' or name.startswith('pydantic'):
        sys.modules.pop(name, None)

import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

mlflow.set_experiment("/Shared/tech-challenge-fase1/notebooks/churn_prediction")

with mlflow.start_run(run_name="logistic_regression_baseline"):
    model = LogisticRegression(random_state=42)
    model.fit(X_train, y_train)

    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    train_accuracy = accuracy_score(y_train, y_pred_train)
    test_accuracy = accuracy_score(y_test, y_pred_test)
    test_f1 = f1_score(y_test, y_pred_test)
    test_precision = precision_score(y_test, y_pred_test)
    test_recall = recall_score(y_test, y_pred_test)

    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("test_f1_score", test_f1)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("test_recall", test_recall)

    overfitting = train_accuracy - test_accuracy
    mlflow.log_metric("overfitting", overfitting)

    mlflow.sklearn.log_model(model, "model")

    print(f"=== LOGISTIC REGRESSION ===")
    print(f"Train Accuracy: {train_accuracy:.4f}")
    print(f"Test Accuracy:  {test_accuracy:.4f}")
    print(f"Test F1 Score:  {test_f1:.4f}")

    print(f"Test Precision: {test_precision:.4f}")
    print(f"Test Recall:    {test_recall:.4f}")
    print(f"Overfitting:    {overfitting:.4f}")

## 6. Persistir dataframe pré-processado

In [0]:
# Persistir o dataframe pré-processado
df_encoded.to_csv("../data/pre-processed/Telco_customer_churn_preprocessed.csv", index=False)

## 7. Persistir modelo

In [0]:
# Persistir o modelo treinado com MLFlow
mlflow.sklearn.log_model(model, "logistic_regression_model")

# Persistir o modelo usando joblib
import joblib
joblib.dump(model, "../models/baseline_model.joblib")